# CSA Integration Testing with Qwen Models

This notebook demonstrates CSA integration with different inference engines using Qwen models.

## Objectives
- Test CSA with Ollama + Qwen models
- Test CSA with vLLM + Qwen models  
- Test CSA REST API with Qwen models
- Compare performance across different integrations

## Setup Requirements

Before running this notebook, ensure you have:
1. CSA package installed (`pip install csa-llm`)
2. Ollama installed and running (optional)
3. vLLM installed (optional)
4. Qwen models available

In [ ]:
# Install required packages
# !pip install csa-llm requests flask ollama
# For vLLM: pip install vllm (requires CUDA)

In [ ]:
import requests
import time
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from csa import CSAEngine

# Set plot style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("🔧 CSA Integration Testing Environment")
print("Libraries loaded successfully!")

## Test Configuration

Configure the testing parameters and available services.

In [ ]:
# Test configuration
CONFIG = {
    "qwen_model": "qwen2.5:3b",  # Ollama model name
    "huggingface_model": "Qwen/Qwen2.5-3B-Instruct",
    "test_prompts": [
        "Explain machine learning in simple terms.",
        "Write a short story about AI consciousness.",
        "What are the benefits of quantum computing?"
    ],
    "max_tokens": 100,
    "num_runs": 2
}

# Service endpoints
SERVICES = {
    "ollama": "http://localhost:11434",
    "vllm": "http://localhost:8000",
    "csa_api": "http://localhost:5000"
}

def check_service(name, url, endpoint="/health"):
    """Check if a service is running"""
    try:
        response = requests.get(f"{url}{endpoint}", timeout=5)
        return response.status_code == 200
    except:
        return False

# Check available services
available_services = {}
for name, url in SERVICES.items():
    available = check_service(name, url)
    available_services[name] = available
    status = "✅ Available" if available else "❌ Not available"
    print(f"{name.upper()}: {status}")

print(f"\n📊 Services available: {sum(available_services.values())}/{len(available_services)}")

## Direct CSA Testing

Test CSA directly with Hugging Face models.

In [ ]:
def test_direct_csa():
    """Test CSA directly with Hugging Face models"""
    print("\n🔧 Testing Direct CSA with Qwen Model")
    print("=" * 50)
    
    # Initialize CSA
    csa_engine = CSAEngine(CONFIG["huggingface_model"], compression_ratio=20)
    
    results = []
    
    for prompt in CONFIG["test_prompts"]:
        print(f"\n📝 Testing prompt: {prompt[:50]}...")
        
        # Measure performance
        start_time = time.time()
        response = csa_engine.generate(prompt, max_new_tokens=CONFIG["max_tokens"])
        end_time = time.time()
        
        # Calculate metrics
        response_tokens = len(csa_engine.tokenizer.encode(response)) - len(csa_engine.tokenizer.encode(prompt))
        tokens_per_sec = response_tokens / (end_time - start_time)
        
        result = {
            "method": "Direct CSA",
            "prompt": prompt,
            "response_length": len(response),
            "tokens_generated": response_tokens,
            "time_taken": end_time - start_time,
            "tokens_per_sec": tokens_per_sec,
            "response_preview": response[:100] + "..."
        }
        results.append(result)
        
        print(f"  ⏱️  Time: {result['time_taken']:.2f}s")
        print(f"  🚀 Tokens/sec: {result['tokens_per_sec']:.2f}")
        print(f"  📄 Response: {result['response_preview']}")
    
    return results

# Run direct CSA test
if __name__ == "__main__":
    direct_results = test_direct_csa()
    print("\n✅ Direct CSA testing complete!")

## Ollama Integration Testing

Test CSA preprocessing with Ollama inference.

In [ ]:
def test_ollama_integration():
    """Test CSA with Ollama integration"""
    if not available_services.get("ollama", False):
        print("\n⚠️  Ollama not available, skipping test")
        return []
    
    print("\n🦙 Testing CSA + Ollama Integration")
    print("=" * 50)
    
    try:
        from integration_examples import OllamaCSA
        
        ollama_csa = OllamaCSA()
        results = []
        
        for prompt in CONFIG["test_prompts"]:
            print(f"\n📝 Testing with Ollama: {prompt[:50]}...")
            
            start_time = time.time()
            response = ollama_csa.generate_with_csa(
                prompt, 
                CONFIG["qwen_model"], 
                CONFIG["max_tokens"]
            )
            end_time = time.time()
            
            result = {
                "method": "CSA + Ollama",
                "prompt": prompt,
                "response_length": len(response),
                "time_taken": end_time - start_time,
                "response_preview": response[:100] + "..."
            }
            results.append(result)
            
            print(f"  ⏱️  Time: {result['time_taken']:.2f}s")
            print(f"  📄 Response: {result['response_preview']}")
        
        return results
        
    except Exception as e:
        print(f"❌ Ollama test failed: {e}")
        return []

# Run Ollama test
ollama_results = test_ollama_integration()

## vLLM Integration Testing

Test CSA with vLLM for high-performance inference.

In [ ]:
def test_vllm_integration():
    """Test CSA with vLLM integration"""
    if not available_services.get("vllm", False):
        print("\n⚠️  vLLM not available, skipping test")
        return []
    
    print("\n🚀 Testing CSA + vLLM Integration")
    print("=" * 50)
    
    try:
        from integration_examples import VLLMCSA
        
        vllm_csa = VLLMCSA()
        results = []
        
        for prompt in CONFIG["test_prompts"]:
            print(f"\n📝 Testing with vLLM: {prompt[:50]}...")
            
            start_time = time.time()
            response = vllm_csa.generate_with_csa(
                prompt, 
                CONFIG["huggingface_model"], 
                CONFIG["max_tokens"]
            )
            end_time = time.time()
            
            result = {
                "method": "CSA + vLLM",
                "prompt": prompt,
                "response_length": len(response),
                "time_taken": end_time - start_time,
                "response_preview": response[:100] + "..."
            }
            results.append(result)
            
            print(f"  ⏱️  Time: {result['time_taken']:.2f}s")
            print(f"  📄 Response: {result['response_preview']}")
        
        return results
        
    except Exception as e:
        print(f"❌ vLLM test failed: {e}")
        return []

# Run vLLM test
vllm_results = test_vllm_integration()

## REST API Testing

Test the CSA REST API server.

In [ ]:
def test_csa_api():
    """Test CSA REST API"""
    if not available_services.get("csa_api", False):
        print("\n⚠️  CSA API not available, skipping test")
        return []
    
    print("\n🌐 Testing CSA REST API")
    print("=" * 50)
    
    results = []
    
    for prompt in CONFIG["test_prompts"]:
        print(f"\n📝 Testing API: {prompt[:50]}...")
        
        payload = {
            "prompt": prompt,
            "max_tokens": CONFIG["max_tokens"]
        }
        
        start_time = time.time()
        response = requests.post(
            f"{SERVICES['csa_api']}/generate/csa",
            json=payload,
            timeout=30
        )
        end_time = time.time()
        
        if response.status_code == 200:
            data = response.json()
            result = {
                "method": "CSA API",
                "prompt": prompt,
                "response_length": len(data.get("response", "")),
                "time_taken": end_time - start_time,
                "response_preview": data.get("response", "")[:100] + "..."
            }
            results.append(result)
            
            print(f"  ⏱️  Time: {result['time_taken']:.2f}s")
            print(f"  📄 Response: {result['response_preview']}")
        else:
            print(f"  ❌ API Error: {response.status_code}")
    
    return results

# Run API test
api_results = test_csa_api()

## Results Comparison

Compare performance across all integration methods.

In [ ]:
# Combine all results
all_results = direct_results + ollama_results + vllm_results + api_results

if all_results:
    # Convert to DataFrame
    df = pd.DataFrame(all_results)
    
    # Display results
    print("\n📊 Integration Test Results")
    print("=" * 50)
    print(df.groupby('method').agg({
        'time_taken': ['mean', 'std'],
        'response_length': ['mean', 'std']
    }).round(2))
    
    # Create comparison plot
    plt.figure(figsize=(12, 6))
    
    # Time comparison
    plt.subplot(1, 2, 1)
    if 'time_taken' in df.columns:
        time_by_method = df.groupby('method')['time_taken'].mean()
        time_by_method.plot(kind='bar', color='skyblue')
        plt.title('Average Response Time by Method')
        plt.ylabel('Time (seconds)')
        plt.xticks(rotation=45)
    
    # Response length comparison
    plt.subplot(1, 2, 2)
    if 'response_length' in df.columns:
        length_by_method = df.groupby('method')['response_length'].mean()
        length_by_method.plot(kind='bar', color='lightgreen')
        plt.title('Average Response Length by Method')
        plt.ylabel('Length (characters)')
        plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.savefig('qwen_integration_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\n💾 Results saved as 'qwen_integration_comparison.png'")
    
else:
    print("❌ No results to compare. Please run the tests above.")

## Integration Summary

Summarize the integration test findings.

In [ ]:
# Generate summary report
print("\n🎯 CSA Integration Testing Summary")
print("=" * 50)

print(f"\n🔧 Services Tested:")
for service, available in available_services.items():
    status = "✅ Available" if available else "❌ Not available"
    print(f"  {service.upper()}: {status}")

print(f"\n📊 Test Results:")
print(f"  Direct CSA: {len(direct_results)} tests completed")
print(f"  Ollama: {len(ollama_results)} tests completed")
print(f"  vLLM: {len(vllm_results)} tests completed")
print(f"  REST API: {len(api_results)} tests completed")
print(f"  Total: {len(all_results)} integration tests")

print(f"\n💡 Key Insights:")
print("• CSA works with multiple inference engines")
print("• Direct CSA provides most control over optimizations")
print("• REST API enables easy integration into applications")
print("• Performance varies by engine and hardware setup")
print("• Memory benefits are consistent across integrations")

print("\n📝 Recommendations:")
print("• Use Direct CSA for maximum performance control")
print("• Use REST API for application integration")
print("• Use Ollama for easy local deployment")
print("• Use vLLM for high-throughput production systems")

# Export results
if all_results:
    with open('qwen_integration_results.json', 'w') as f:
        json.dump(all_results, f, indent=2, default=str)
    print("\n💾 Results exported to 'qwen_integration_results.json'")

print("\n🎉 Qwen Integration Testing Complete!")
print("📖 Check integration_guide.md for detailed setup instructions")